In [1]:
!pip install pyspark

In [2]:

from pyspark.sql import SparkSession

# Crear sesión de Spark
spark = SparkSession.builder \
    .appName("DataSalud_Peru") \
    .getOrCreate()

print("Apache Spark funciona correctamente")
print("Version de Spark:", spark.version)


Apache Spark funciona correctamente
Version de Spark: 4.0.5


In [3]:

datos_prueba = [
    (1, 100),
    (2, 200),
    (3, 150),
    (4, 300)
]

df_prueba = spark.createDataFrame(
    datos_prueba,
    ["TiempoKey", "Casos"]
)

df_prueba.show()


+---------+-----+
|TiempoKey|Casos|
+---------+-----+
|        1|  100|
|        2|  200|
|        3|  150|
|        4|  300|
+---------+-----+



In [4]:

from google.colab import files

archivos = files.upload()


Saving fact_casos_neumonia_spark.csv to fact_casos_neumonia_spark.csv


In [9]:
import time
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType

# Definir el esquema explícito ya que el CSV no contiene cabecera
esquema = StructType([
    StructField("TiempoKey", IntegerType(), True),
    StructField("Casos", IntegerType(), True)
])

# Leer el archivo CSV sin cabecera, con el separador correcto ';'
df = spark.read.csv(
    "fact_casos_neumonia_spark.csv",
    header=False,
    schema=esquema,
    sep=";"
)

# Mostrar estructura
print("ESTRUCTURA DEL DATASET")
df.printSchema()

# Mostrar registros
print("PRIMEROS REGISTROS")
df.show(10)

# Contar registros
total_registros = df.count()

print("Total de registros:", total_registros)

# Agregar los casos por periodo de tiempo
inicio = time.perf_counter()

resultado_df = (
    df.groupBy("TiempoKey")
      .agg(F.sum("Casos").alias("Total_Casos"))
      .orderBy("TiempoKey")
)

# Ejecutar y materializar el resultado
resultado_df.show(50, truncate=False)

tiempo_dataframe = time.perf_counter() - inicio

print("Tiempo de procesamiento DataFrame:",
      round(tiempo_dataframe * 1000, 3), "ms")

ESTRUCTURA DEL DATASET
root
 |-- TiempoKey: integer (nullable = true)
 |-- Casos: integer (nullable = true)

PRIMEROS REGISTROS
+---------+-----+
|TiempoKey|Casos|
+---------+-----+
|   202601|    0|
|   202601|    1|
|   202601|    0|
|   202601|    0|
|   202601|    1|
|   202601|    4|
|   202601|    0|
|   202601|    0|
|   202601|    0|
|   202601|    0|
+---------+-----+
only showing top 10 rows
Total de registros: 73020
+---------+-----------+
|TiempoKey|Total_Casos|
+---------+-----------+
|202601   |156        |
|202602   |156        |
|202603   |152        |
|202604   |138        |
|202605   |139        |
|202606   |145        |
|202607   |157        |
|202608   |167        |
|202609   |297        |
|202610   |336        |
|202611   |281        |
|202612   |281        |
|202613   |254        |
|202614   |261        |
|202615   |219        |
|202616   |240        |
|202617   |168        |
|202618   |146        |
|202619   |201        |
|202620   |230        |
|202621   |207   

In [10]:

# Convertir DataFrame a RDD
rdd = df.select("TiempoKey", "Casos").rdd

inicio = time.perf_counter()

# Agrupar y sumar los casos por periodo
resultado_rdd = (
    rdd.map(lambda fila: (fila["TiempoKey"], fila["Casos"]))
       .reduceByKey(lambda a, b: a + b)
       .sortByKey()
)

# Mostrar resultados
print(resultado_rdd.collect())

tiempo_rdd = time.perf_counter() - inicio

print("Tiempo de procesamiento RDD:",
      round(tiempo_rdd * 1000, 3), "ms")


[(202601, 156), (202602, 156), (202603, 152), (202604, 138), (202605, 139), (202606, 145), (202607, 157), (202608, 167), (202609, 297), (202610, 336), (202611, 281), (202612, 281), (202613, 254), (202614, 261), (202615, 219), (202616, 240), (202617, 168), (202618, 146), (202619, 201), (202620, 230), (202621, 207), (202622, 159), (202623, 161), (202624, 128), (202625, 142), (202626, 151), (202627, 153), (202628, 191), (202629, 143), (202630, 110), (202631, 114), (202632, 102), (202633, 128), (202634, 125), (202635, 92)]
Tiempo de procesamiento RDD: 1838.447 ms


In [11]:

# Comparar los resultados de DataFrame y RDD

resultado_df_lista = [
    (fila["TiempoKey"], fila["Total_Casos"])
    for fila in resultado_df.collect()
]

resultado_rdd_lista = resultado_rdd.collect()

if resultado_df_lista == resultado_rdd_lista:
    print("CORRECTO: Ambos resultados coinciden.")
else:
    print("ADVERTENCIA: Los resultados son diferentes.")


CORRECTO: Ambos resultados coinciden.


In [12]:

import time
import statistics

# Calentamiento inicial
resultado_df.collect()
resultado_rdd.collect()

# Medir DataFrame cinco veces
tiempos_df = []

for i in range(5):
    inicio = time.perf_counter()
    resultado_df.collect()
    tiempos_df.append((time.perf_counter() - inicio) * 1000)

# Medir RDD cinco veces
tiempos_rdd = []

for i in range(5):
    inicio = time.perf_counter()
    resultado_rdd.collect()
    tiempos_rdd.append((time.perf_counter() - inicio) * 1000)

# Calcular medianas
mediana_df = statistics.median(tiempos_df)
mediana_rdd = statistics.median(tiempos_rdd)

print("RESULTADOS DE LAS 5 PRUEBAS")
print("DataFrame:", [round(x, 2) for x in tiempos_df])
print("RDD:", [round(x, 2) for x in tiempos_rdd])

print("\nMEDIANAS")
print("DataFrame:", round(mediana_df, 2), "ms")
print("RDD:", round(mediana_rdd, 2), "ms")


RESULTADOS DE LAS 5 PRUEBAS
DataFrame: [144.64, 402.62, 462.5, 305.09, 146.62]
RDD: [141.21, 131.74, 114.84, 121.43, 153.96]

MEDIANAS
DataFrame: 305.09 ms
RDD: 131.74 ms
